# Merge NIH and VinBig Training Annotations

This notebook combines processed NIH and VinBig bounding boxes into one shared training CSV. It standardizes column names, converts NIH width/height boxes into corner coordinates, records the source dataset, and verifies the merged label distribution.


## Setup

Import CSV/path helpers and `Counter` for the final merged label-count check.


In [2]:
# Notebook guide: shared imports for merging and verification.
# `Counter` is needed by the final class-count sanity check.

from pathlib import Path
import csv
from collections import Counter


## Standardize and Merge Rows

Read both processed datasets, convert them to the same schema, and write one merged annotation file.


In [3]:
# Notebook guide: merge NIH and VinBig into one normalized CSV.
# NIH boxes are converted from x/y/width/height, while VinBig boxes already use x_min/y_min/x_max/y_max.

nih_csv_path = Path("../data/processed/02_nih_train_selected_classes_pa_only.csv")
vinbig_csv_path = Path("../data/processed/03_vinbig_wbf_bounding_boxes_1024.csv")
output_csv_path = Path("../data/processed/01_merged_nih_vinbig_training_boxes.csv")
output_csv_path.parent.mkdir(parents=True, exist_ok=True)

# Shared schema written for both NIH and VinBig rows.
output_fieldnames = [
    "source",
    "image_id",
    "label_name",
    "x_min",
    "y_min",
    "x_max",
    "y_max",
]

def clean_float(value):
    """Return a clean numeric string, or empty string if missing."""
    value = (value or "").strip()
    if value == "":
        return ""
    return str(round(float(value), 3))

def get_nih_box(row):
    """
    NIH BBox_List format usually stores:
    - Bbox [x -> x_min
    - y -> y_min
    - w -> width
    - h -> height

    In your CSV, the header may appear as:
    Image Index,Finding Label,Bbox [x,y,w,h],,,
    so DictReader names the blank columns as '', None, etc.
    This helper safely reads the first 4 numeric values after Finding Label.
    """
    values = list(row.values())

    x_min = float(values[2])
    y_min = float(values[3])
    width = float(values[4])
    height = float(values[5])

    return x_min, y_min, x_min + width, y_min + height

total_nih_rows = 0
kept_nih_rows = 0
total_vinbig_rows = 0
kept_vinbig_rows = 0

with output_csv_path.open("w", newline="", encoding="utf-8") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=output_fieldnames)
    writer.writeheader()

    # Add NIH rows after converting box format.
    with nih_csv_path.open(newline="", encoding="utf-8") as nih_file:
        reader = csv.DictReader(nih_file)

        for row in reader:
            total_nih_rows += 1

            try:
                x_min, y_min, x_max, y_max = get_nih_box(row)
            except (ValueError, IndexError, TypeError):
                # Skip malformed NIH rows instead of stopping the whole merge.
                continue

            writer.writerow({
                "source": "NIH",
                "image_id": (row.get("Image Index") or "").strip(),
                "label_name": (row.get("Finding Label") or "").strip(),
                "x_min": round(x_min, 3),
                "y_min": round(y_min, 3),
                "x_max": round(x_max, 3),
                "y_max": round(y_max, 3),
            })
            kept_nih_rows += 1

    # Add VinBig rows; WBF output is already in corner-coordinate format.
    with vinbig_csv_path.open(newline="", encoding="utf-8") as vinbig_file:
        reader = csv.DictReader(vinbig_file)

        for row in reader:
            total_vinbig_rows += 1

            try:
                writer.writerow({
                    "source": "VinBig",
                    "image_id": (row.get("image_id") or "").strip(),
                    "label_name": (row.get("label_name") or row.get("class_name") or "").strip(),
                    "x_min": clean_float(row.get("x_min")),
                    "y_min": clean_float(row.get("y_min")),
                    "x_max": clean_float(row.get("x_max")),
                    "y_max": clean_float(row.get("y_max")),
                })
                kept_vinbig_rows += 1
            except ValueError:
                # Skip rows with non-numeric coordinates.
                continue

print(f"NIH rows read: {total_nih_rows:,}")
print(f"NIH rows written: {kept_nih_rows:,}")
print(f"VinBig rows read: {total_vinbig_rows:,}")
print(f"VinBig rows written: {kept_vinbig_rows:,}")
print(f"Merged rows written: {kept_nih_rows + kept_vinbig_rows:,}")
print(f"Output CSV: {output_csv_path.resolve()}")


NIH rows read: 364
NIH rows written: 364
VinBig rows read: 15,116
VinBig rows written: 15,116
Merged rows written: 15,480
Output CSV: D:\Home\Documents\GitHub\cxraide-data-pipeline\notebooks\data\processed\01_merged_nih_vinbig_training_boxes.csv


## Verify Merged Counts

Count labels in the combined output so the final training CSV can be checked before model preparation.


In [3]:
# Notebook guide: verify the class distribution in the merged training annotations.
# This should reflect the combined output consumed by downstream notebooks.

bbox_csv_path = Path('../data/processed/01_merged_nih_vinbig_training_boxes.csv')

with bbox_csv_path.open(newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    abnormality_counts = Counter(row['label_name'] for row in reader)

print(f'Total annotated abnormalities: {sum(abnormality_counts.values()):,}')
print('\nAbnormalities per classification:')
for finding_label, count in abnormality_counts.most_common():
    print(f'{finding_label}: {count:,}')


Total annotated abnormalities: 15,480

Abnormalities per classification:
Pleural thickening: 4,056
Pulmonary fibrosis: 3,364
Cardiomegaly: 2,378
Nodule/Mass: 1,865
Pleural effusion: 1,855
Infiltration: 1,002
Consolidation: 434
Atelectasis: 331
Pneumothorax: 195
